In [1]:
from pathlib import Path

# Find repo root
REPO_ROOT = Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

REPORT_ROOT = REPO_ROOT / "report"

FIGSIZE = (20,18)
DPI = 100
GENERATE_PLOTS = False

Repo root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/4. Semester/DEDA Project/DEDA_LLM_Spatial_Hotelling


In [2]:
import pandas as pd
import geopandas as gpd
import sys
import json
from shapely.geometry import shape
from hotelling.spatial.admin import join_lor_names
from hotelling.spatial.census import make_cell_id

# Find repo root
REPO_ROOT = Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

REPORT_ROOT = REPO_ROOT / "report"

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from hotelling.spatial.boundaries import load_boundary

PATH_RAW = REPO_ROOT / Path('data/raw')
PATH_PROCESSED = REPO_ROOT / Path('data/processed')

# Midpoint table (center coordinates)
zensus = gpd.read_parquet(PATH_RAW / 'zensus2022_grid.parquet')
zensus_filtered = gpd.read_parquet(PATH_RAW / 'zensus2022_grid_filtered.parquet')
lor = gpd.read_parquet(PATH_PROCESSED / 'lor.parquet')

# CRITICAL FIX: berlin.geojson has EPSG:3035 coordinates but geopandas auto-detects as EPSG:4326
# We must force the correct CRS instead of transforming from the wrong one
with open(PATH_RAW / 'city_boundary_Berlin.geojson', 'r') as f:
    berlin_json = json.load(f)
berlin = gpd.GeoDataFrame([1], geometry=[shape(berlin_json['geometry'])], crs='EPSG:3035')

boundary = load_boundary(PATH_RAW / 'relation_boundary_14983.geojson')

# Load pop_grid

grid = gpd.read_parquet(PATH_PROCESSED / 'pop_grid.parquet')

# Build squares from points of grid
grid['geometry'] = grid.apply(lambda row: row.geometry.buffer(50, cap_style='square'), axis=1)
grid['index'] = grid.index

Repo root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/4. Semester/DEDA Project/DEDA_LLM_Spatial_Hotelling


In [3]:
# Load the OSM supermarkets data
osm_supermarkets = gpd.read_parquet(PATH_PROCESSED / 'supermarkets.parquet')

In [4]:
# Load gtfs data for public transport stops
stops = pd.read_csv(PATH_RAW / 'gtfs' / 'stops.txt')
stops = gpd.GeoDataFrame(
    stops,
    geometry=gpd.points_from_xy(stops.stop_lon, stops.stop_lat),
    crs='EPSG:4326'
).to_crs('EPSG:3035')

# Include only stops in within Berlin
# stops = stops[stops.intersects(berlin.geometry[0])]


In [5]:
# Load the rest of the data
routes     = pd.read_csv(PATH_RAW / 'gtfs' / "routes.txt")
trips      = pd.read_csv(PATH_RAW / 'gtfs' / "trips.txt")
stop_times = pd.read_csv(PATH_RAW / 'gtfs' / "stop_times.txt")

/var/folders/y8/4_9g68pj7k136q2yypgp5ysc0000gn/T/ipykernel_5822/3273831377.py:3: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  trips      = pd.read_csv(PATH_RAW / 'gtfs' / "trips.txt")
/var/folders/y8/4_9g68pj7k136q2yypgp5ysc0000gn/T/ipykernel_5822/3273831377.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  stop_times = pd.read_csv(PATH_RAW / 'gtfs' / "stop_times.txt")


In [6]:
# One representative trip per route+direction (pick first)
rep_trips = (
    trips.groupby(["route_id", "direction_id"], as_index=False)
         .first()[["route_id", "direction_id", "trip_id"]]
)

# Join: route → trip → stop_times → stop coords
seq = (
    rep_trips
    .merge(stop_times[["trip_id", "stop_id", "stop_sequence"]], on="trip_id")
    .merge(stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]], on="stop_id")
    .merge(routes[["route_id", "route_short_name", "route_type"]], on="route_id")
    .sort_values(["route_id", "direction_id", "stop_sequence"])
)

In [7]:
seq.to_csv(PATH_PROCESSED / 'representative_routes.csv', index=False)

In [8]:
seq

,route_id,direction_id,trip_id,stop_id,stop_sequence,stop_name,stop_lat,stop_lon,route_short_name,route_type
0,10141_109,0,247802165,de:11000:900053301:2:52,0,S Wannsee Bhf (Berlin),52.421546,13.180177,S1,109
1,10141_109,0,247802165,de:11000:900052201:1:51,1,S Nikolassee (Berlin),52.431741,13.194195,S1,109
2,10141_109,0,247802165,de:11000:900050355:1:51,2,S Schlachtensee (Berlin),52.440069,13.215529,S1,109
3,10141_109,0,247802165,de:11000:900050301:1:51,3,S Mexikoplatz (Berlin),52.436719,13.233192,S1,109
4,10141_109,0,247802165,de:11000:900049201:1:51,4,S Zehlendorf (Berlin),52.430876,13.258140,S1,109
...,...,...,...,...,...,...,...,...,...,...
40878,9813_700,1,236990187,de:12062:900415929::2,26,"Massen, Am Industriepark",51.637194,13.739781,599,700
40879,9813_700,1,236990187,de:12062:900415928::1,27,"Massen, Feuerwehr",51.643178,13.731242,599,700
40880,9813_700,1,236990187,de:12062:900415142::2,28,"Massen, Schule",51.640436,13.730069,599,700
40881,9813_700,1,236990187,de:12062:900415004::2,29,"Finsterwalde, Cottbuser Str.",51.635499,13.722040,599,700


In [9]:
import matplotlib.pyplot as plt
import contextily as ctx

if GENERATE_PLOTS:
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    stops[stops['parent_station'].isna()].plot(ax=ax, color='royalblue', markersize=1, label='Public Transport Stops', alpha=0.5)
    berlin.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=2, label='Berlin Boundary')
    ctx.add_basemap(ax, crs=berlin.crs, source=ctx.providers.OpenStreetMap.Mapnik, zoom=11, zorder=0, alpha = 0.2)
    ax.set_axis_off()
    ax.legend()
    plt.title('Public Transport Stops in Berlin', fontsize=16)
    plt.show()


In [10]:
grid['cell_id'] = grid.apply(make_cell_id, axis=1)

In [11]:
import partridge as ptg  # pip install partridge

# Bounding box: inner Ring + buffer, in WGS84
BBOX = tuple(berlin.to_crs("EPSG:4326").total_bounds)  # (min_lon, min_lat, max_lon, max_lat)

view = {
    "stops.txt": {
        "stop_lat": lambda x: (x >= BBOX[1]) & (x <= BBOX[3]),
        "stop_lon": lambda x: (x >= BBOX[0]) & (x <= BBOX[2]),
    }
}
feed = ptg.load_feed(str(PATH_RAW / "gtfs"), view=view)
# Then write trimmed feed back out
ptg.writers.write_feed_dangerously(feed, str(PATH_RAW / "gtfs_berlin"))

'/Users/jedrek/Documents/Studium Volkswirschaftslehre/4. Semester/DEDA Project/DEDA_LLM_Spatial_Hotelling/data/raw/gtfs_berlin.zip'

In [12]:
from hotelling.spatial.distance import build_transit_travel_times

# build_transit_travel_times handles:
#   - GTFS zip construction (skipping empty files)
#   - r5py TransportNetwork creation
#   - TravelTimeMatrix computation
#   - Saving data/processed/travel_times.parquet
travel_times = build_transit_travel_times(
    grid=grid,
    supermarkets=osm_supermarkets,
    # Paths are resolved automatically from repo root.
    # Override if needed:
    # osm_pbf_path=PATH_RAW / "berlin-260512.osm.pbf",
    # gtfs_dir=PATH_RAW / "gtfs",
    # gtfs_zip=PATH_RAW / "gtfs_berlin.zip",
    # output_path=PATH_PROCESSED / "travel_times.parquet",
)

In [13]:
# travel_times.parquet is saved automatically by build_transit_travel_times.
# Load it back for verification:
travel_times = pd.read_parquet(PATH_PROCESSED / "travel_times.parquet")

In [14]:
travel_times = pd.read_parquet(PATH_PROCESSED / "travel_times.parquet")

In [15]:
grid_travel_times = grid.merge(
    travel_times[["from_id", "to_id", "travel_time"]],
    left_on="cell_id",
    right_on="from_id",
    how="left"
)
grid_travel_times['if_pop'] = grid_travel_times['Einwohner'] > 0

In [16]:
osm_supermarkets

,geometry,name,chain,chain_type
0,POINT (4552250.633 3272143.971),Netto Marken-Discount,Netto Marken-Discount,discount
1,POINT (4546615.209 3269830.694),nah und gut,Edeka,standard
2,POINT (4548009.708 3276693.622),Aldi,Aldi Nord,discount
3,POINT (4555261.291 3276925.952),Netto,Netto,discount
4,POINT (4556560.998 3272374.872),EDEKA Thaut,Edeka,standard
...,...,...,...,...
489,POINT (4547453.764 3266616.203),EDEKA Groß,Edeka,standard
490,POINT (4552840.905 3273147.549),EDEKA,Edeka,standard
491,POINT (4550918.827 3276538.696),REWE Center,Rewe,standard
492,POINT (4547848.602 3274020.546),Kaufland,Kaufland,standard


In [17]:
osm_supermarkets
# ID: 491
# (grid_travel_times[grid_travel_times['to_id'] == STORE_ID]['travel_time'].isna())
import numpy as np
STORE_ID = np.random.choice(destinations['id'])
if GENERATE_PLOTS:
    fig, ax = plt.subplots(figsize=(20,18), dpi=300)
    grid_travel_times[grid_travel_times['to_id'] == STORE_ID].plot(ax=ax, column='travel_time', cmap='viridis', legend=True, alpha =0.5)
    grid_travel_times[(~grid_travel_times['if_pop']) & (grid_travel_times['to_id'] == STORE_ID) & (grid_travel_times[grid_travel_times['to_id'] == STORE_ID]['travel_time'].isna())].plot(ax=ax, color='red', legend=True, alpha =0.5)
    destinations[destinations['id'] == STORE_ID].to_crs(berlin.crs).plot(ax=ax, color='yellow', markersize=5, label=f'Supermarket {STORE_ID}')
    berlin.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=2, label='Berlin Boundary')
    ctx.add_basemap(ax, crs=berlin.crs, source=ctx.providers.OpenStreetMap.Mapnik, zoom=11, zorder=0, alpha = 0.2)
    ax.set_axis_off()
    ax.legend()
    plt.title(f'Travel Times to Supermarket {STORE_ID}', fontsize=16)
    plt.show()

NameError: name 'destinations' is not defined

In [ ]:
if not GENERATE_PLOTS:
    import nbformat, pathlib

    _nb_path = pathlib.Path(__file__) if "__file__" in dir() else None
    # Fallback: set explicitly if auto-detection unavailable
    _nb_path = pathlib.Path("GEO_05_dist_matrix.ipynb")  # ← set once per notebook

    _nb = nbformat.read(_nb_path, as_version=4)
    for _cell in _nb.cells:
        _cell["outputs"] = []
        _cell["execution_count"] = None
    nbformat.write(_nb, _nb_path)
    print(f"Outputs cleared: {_nb_path.name}")